In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

PROJECT_DIR = "/content/drive/MyDrive/Cancer Evolution Arena"
DATA_DIR = f"{PROJECT_DIR}/Data"
LUAD_DIR = f"{DATA_DIR}/luad_tcga_pub"

mutation_path = f"{LUAD_DIR}/data_mutations.txt"
clinical_path = f"{LUAD_DIR}/data_clinical_patient.txt"

print(os.path.exists(mutation_path))
print(os.path.exists(clinical_path))

Mounted at /content/drive
True
True


In [3]:
df = pd.read_csv(
    mutation_path,
    sep="\t",
    comment="#",
    low_memory=False
)

patient_gene = pd.crosstab(
    df["Tumor_Sample_Barcode"],
    df["Hugo_Symbol"]
)

patient_gene = (patient_gene > 0).astype(int)

patient_gene.index = [x[:12] for x in patient_gene.index]

print(patient_gene.shape)
patient_gene.head()

(230, 15130)


Hugo_Symbol,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11B,ZYX,ZZEF1,ZZZ3,snoU13
TCGA-05-4249,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TCGA-05-4382,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TCGA-05-4384,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TCGA-05-4389,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TCGA-05-4390,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [4]:
cancer_genes_v2 = [
    "TP53","KRAS","EGFR","STK11","KEAP1",
    "NF1","RB1","MET","PIK3CA","BRAF",
    "ALK","RET","ROS1","ERBB4","ERBB2",
    "NTRK1","NTRK2","NTRK3","FGFR1","FGFR2",
    "FGFR3","FGFR4","ATM","ATR","ATRX",
    "CDKN2A","CCND1","MYC","PTEN","AKT1",
    "AKT2","AKT3","MAP2K1","MAP2K2","MAPK1",
    "NOTCH1","NOTCH2","NOTCH3","SMARCA4","SETD2",
    "RBM10","CTNNB1","DDR2","JAK1","STAT3",
    "STAT5B","RIT1","RAF1","TSC1","TSC2"
]

available_genes = [
    g for g in cancer_genes_v2
    if g in patient_gene.columns
]

print("Available genes:", len(available_genes))
print(available_genes)

Available genes: 50
['TP53', 'KRAS', 'EGFR', 'STK11', 'KEAP1', 'NF1', 'RB1', 'MET', 'PIK3CA', 'BRAF', 'ALK', 'RET', 'ROS1', 'ERBB4', 'ERBB2', 'NTRK1', 'NTRK2', 'NTRK3', 'FGFR1', 'FGFR2', 'FGFR3', 'FGFR4', 'ATM', 'ATR', 'ATRX', 'CDKN2A', 'CCND1', 'MYC', 'PTEN', 'AKT1', 'AKT2', 'AKT3', 'MAP2K1', 'MAP2K2', 'MAPK1', 'NOTCH1', 'NOTCH2', 'NOTCH3', 'SMARCA4', 'SETD2', 'RBM10', 'CTNNB1', 'DDR2', 'JAK1', 'STAT3', 'STAT5B', 'RIT1', 'RAF1', 'TSC1', 'TSC2']


In [5]:
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

patient_gene_res = patient_gene[available_genes]

results = []
genes = list(patient_gene_res.columns)

for i in range(len(genes)):
    for j in range(i + 1, len(genes)):
        gene_a = genes[i]
        gene_b = genes[j]

        a = patient_gene_res[gene_a]
        b = patient_gene_res[gene_b]

        both = ((a == 1) & (b == 1)).sum()
        only_a = ((a == 1) & (b == 0)).sum()
        only_b = ((a == 0) & (b == 1)).sum()
        neither = ((a == 0) & (b == 0)).sum()

        contingency = [[both, only_a], [only_b, neither]]

        odds_ratio, p_value = fisher_exact(contingency)

        results.append({
            "GeneA": gene_a,
            "GeneB": gene_b,
            "CoOccurrence": both,
            "OddsRatio": odds_ratio,
            "PValue": p_value
        })

res_results = pd.DataFrame(results)

res_results["Adj_P"] = multipletests(
    res_results["PValue"],
    method="fdr_bh"
)[1]

res_results.head()

,GeneA,GeneB,CoOccurrence,OddsRatio,PValue,Adj_P
0,TP53,KRAS,25,0.445122,0.007194,0.340127
1,TP53,EGFR,22,2.190045,0.043023,0.822777
2,TP53,STK11,11,0.371408,0.008858,0.340127
3,TP53,KEAP1,14,0.561621,0.119268,0.995564
4,TP53,NF1,21,3.093023,0.006245,0.340127


In [6]:
res_sig = res_results[
    res_results["Adj_P"] < 0.05
].copy()

print("Significant edges:", len(res_sig))

res_sig.sort_values("Adj_P").head(30)

Significant edges: 1


,GeneA,GeneB,CoOccurrence,OddsRatio,PValue,Adj_P
447,ALK,ERBB4,11,14.895833,5.189184e-07,0.000636


In [7]:
cooccurring_edges = res_sig[
    res_sig["OddsRatio"] > 1
].sort_values("OddsRatio", ascending=False)

exclusive_edges = res_sig[
    res_sig["OddsRatio"] < 1
].sort_values("OddsRatio", ascending=True)

print("Co-occurring edges:", len(cooccurring_edges))
print("Mutually exclusive edges:", len(exclusive_edges))

cooccurring_edges.head(20), exclusive_edges.head(20)

Co-occurring edges: 1
Mutually exclusive edges: 0


(    GeneA  GeneB  CoOccurrence  OddsRatio        PValue     Adj_P
 447   ALK  ERBB4            11  14.895833  5.189184e-07  0.000636,
 Empty DataFrame
 Columns: [GeneA, GeneB, CoOccurrence, OddsRatio, PValue, Adj_P]
 Index: [])

In [8]:
import networkx as nx

G_res = nx.Graph()

for _, row in res_sig.iterrows():
    G_res.add_edge(
        row["GeneA"],
        row["GeneB"],
        odds_ratio=row["OddsRatio"],
        adj_p=row["Adj_P"],
        cooccurrence=row["CoOccurrence"]
    )

print("Nodes:", G_res.number_of_nodes())
print("Edges:", G_res.number_of_edges())
print("Density:", nx.density(G_res))

Nodes: 2
Edges: 1
Density: 1.0


In [9]:
major_genes = ["KRAS", "EGFR", "TP53", "STK11", "KEAP1", "MET", "NF1"]

for gene in major_genes:
    if gene in G_res.nodes:
        neighbors = list(G_res.neighbors(gene))
        print(f"\n{gene} ecosystem:")
        for n in neighbors:
            edge = G_res[gene][n]
            print(
                f"  {n} | OR={edge['odds_ratio']:.2f}, "
                f"Adj_P={edge['adj_p']:.4f}, "
                f"CoOcc={edge['cooccurrence']}"
            )

In [10]:
res_explore = res_results[
    (res_results["PValue"] < 0.05) &
    (
        (res_results["OddsRatio"] > 2) |
        (res_results["OddsRatio"] < 0.5)
    )
].copy()

print("Exploratory edges:", len(res_explore))

res_explore.sort_values("PValue").head(30)

Exploratory edges: 72


,GeneA,GeneB,CoOccurrence,OddsRatio,PValue,Adj_P
447,ALK,ERBB4,11,14.895833,5.189184e-07,0.000636
49,KRAS,EGFR,2,0.101287,1.227048e-04,0.061169
16,TP53,NTRK3,18,8.089888,1.498016e-04,0.061169
52,KRAS,NF1,2,0.124266,6.490371e-04,0.198768
97,EGFR,STK11,0,0.000000,1.123339e-03,0.275218
1144,NOTCH2,RAF1,2,inf,1.708753e-03,0.340127
364,PIK3CA,BRAF,6,6.715909,2.372698e-03,0.340127
960,CCND1,SMARCA4,2,inf,2.961838e-03,0.340127
22,TP53,ATR,10,12.577320,3.296894e-03,0.340127
569,ERBB4,ATRX,7,5.222656,3.378265e-03,0.340127


In [11]:
target_genes = ["KRAS", "EGFR", "STK11", "KEAP1", "TP53", "NF1", "MET", "BRAF", "PIK3CA"]

target_edges = res_results[
    (
        res_results["GeneA"].isin(target_genes) |
        res_results["GeneB"].isin(target_genes)
    )
].copy()

target_edges.sort_values("PValue").head(40)

,GeneA,GeneB,CoOccurrence,OddsRatio,PValue,Adj_P
49,KRAS,EGFR,2,0.101287,0.000123,0.061169
16,TP53,NTRK3,18,8.089888,0.000150,0.061169
52,KRAS,NF1,2,0.124266,0.000649,0.198768
97,EGFR,STK11,0,0.000000,0.001123,0.275218
364,PIK3CA,BRAF,6,6.715909,0.002373,0.340127
22,TP53,ATR,10,12.577320,0.003297,0.340127
50,KRAS,STK11,21,2.783626,0.004966,0.340127
382,PIK3CA,MYC,2,inf,0.005164,0.340127
4,TP53,NF1,21,3.093023,0.006245,0.340127
10,TP53,RET,9,11.204082,0.006689,0.340127


In [12]:
print("Available genes:", len(available_genes))
print("Significant edges:", len(res_sig))
print("Exploratory edges:", len(res_explore))

Available genes: 50
Significant edges: 1
Exploratory edges: 72


In [13]:
res_explore.sort_values("PValue").head(30)

,GeneA,GeneB,CoOccurrence,OddsRatio,PValue,Adj_P
447,ALK,ERBB4,11,14.895833,5.189184e-07,0.000636
49,KRAS,EGFR,2,0.101287,1.227048e-04,0.061169
16,TP53,NTRK3,18,8.089888,1.498016e-04,0.061169
52,KRAS,NF1,2,0.124266,6.490371e-04,0.198768
97,EGFR,STK11,0,0.000000,1.123339e-03,0.275218
1144,NOTCH2,RAF1,2,inf,1.708753e-03,0.340127
364,PIK3CA,BRAF,6,6.715909,2.372698e-03,0.340127
960,CCND1,SMARCA4,2,inf,2.961838e-03,0.340127
22,TP53,ATR,10,12.577320,3.296894e-03,0.340127
569,ERBB4,ATRX,7,5.222656,3.378265e-03,0.340127
